In [1]:
import os,sys, glob
SPARK_HOME = os.environ['SPARK_HOME']

spark_py_zips = glob.glob(f"{SPARK_HOME}/python/lib/*.zip")
sys.path[:0] = spark_py_zips

In [2]:
# Import SparkSession
from pyspark.sql import SparkSession

# Create SparkSession 
spark = (SparkSession.builder \
      .master("local[*]") \
      .appName("SparkByExamples.com") \
      # .master("local-cluster[3,2,1024]") \
      .getOrCreate() 
    )

spark.sparkContext.setLogLevel("WARN")

#ALTERNATIVE METHOD TO MODIFY LOGLEVEL DURING INITIALISATION OF spark
#.config("spark.log.level", "INFO") #we can modify sparkContext properties using config() method of SparkSession.Builder class

'''
The config .master("local-cluster[3,2,1024]") means that we will simulate an actual cluster.
The "cluster" will have 3 worker nodes (ie executors) available, each with 2 cores of 1024 MB memory.
'''


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/01 13:59:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'\nThe config .master("local-cluster[3,2,1024]") means that we will simulate an actual cluster.\nThe "cluster" will have 3 worker nodes (ie executors) available, each with 2 cores of 1024 MB memory.\n'

In [3]:
(spark\
    #.read.csv("clubs.csv", header=True, inferSchema=True)\
    .read.csv("/Users/smish257/Downloads/ddac_pgdl_fixed_encrypted_final.csv", header=False, inferSchema=True)\
        .count()
)

7492

# Using the Dataframes API for Spark

In [3]:
sc = spark.sparkContext
print(sc)

<SparkContext master=local-cluster[3,2,1024] appName=SparkByExamples.com>


In [4]:
df = spark.read.csv('/Users/smish257/Library/CloudStorage/OneDrive-AmericanExpress/Desktop/0226/42_1_AD_267.csv', header=True)

In [ ]:
rdd = df.rdd

rdd.take(5)

In [16]:
prediction_counts_rdd = rdd.map(lambda row: (row['prediction'], 1)) \
                            .reduceByKey(lambda a, b: a + b)
# this creates a tuple of (prediction, 1) for each row, and then reduces it by key (prediction) to get the count of each unique prediction.

for prediction, count_prediction in prediction_counts_rdd.collect():
    print(f"Prediction: {prediction}, Count: {count_prediction}")

Prediction: not_compliant, Count: 98
Prediction: compliant, Count: 136
Prediction: not_applicable, Count: 32
Prediction: None, Count: 1


In [7]:
'''
Using the old-RDD API is unintuitive, involving Functional Programming concepts like map and reduceByKey, and is not optimized for performance.
The newer DataFrame API is more user-friendly, allowing us to perform operations using SQL-like syntax.
'''
import pyspark.sql.functions as F

(df \
    .filter("na_tag = FALSE") \
    .groupBy("prediction").agg(F.count("*").alias("prediction_count")) \
    #.count()
    .explain()
)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[prediction#26], functions=[count(1)])
   +- Exchange hashpartitioning(prediction#26, 200), ENSURE_REQUIREMENTS, [plan_id=34]
      +- HashAggregate(keys=[prediction#26], functions=[partial_count(1)])
         +- Project [prediction#26]
            +- Filter (isnotnull(na_tag#29) AND NOT cast(na_tag#29 as boolean))
               +- FileScan csv [prediction#26,na_tag#29] Batched: false, DataFilters: [isnotnull(na_tag#29), NOT cast(na_tag#29 as boolean)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/smish257/Library/CloudStorage/OneDrive-AmericanExpress/Des..., PartitionFilters: [], PushedFilters: [IsNotNull(na_tag)], ReadSchema: struct<prediction:string,na_tag:string>




In [ ]:
df2 = (df \
    .filter("na_tag = FALSE") \
    .groupBy("prediction").agg(F.count("*").alias("prediction_count")) \
)

df2.show()

In [5]:
df.printSchema()

root
 |-- #: string (nullable = true)
 |-- convo_id: string (nullable = true)
 |-- cnvs_ts: string (nullable = true)
 |-- prediction_nlp: string (nullable = true)
 |-- actual_label: string (nullable = true)
 |-- scenario_id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- clean_text: string (nullable = true)
 |-- redacted_transcript: string (nullable = true)
 |-- prediction: string (nullable = true)
 |-- evidence: string (nullable = true)
 |-- reason: string (nullable = true)
 |-- na_tag: string (nullable = true)



In [12]:
df2 = (
    df \
    .withColumn("cnvs_ts",F.to_timestamp("cnvs_ts", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("call_date", F.dayofmonth("cnvs_ts")) \
    .groupBy("call_date").agg(F.count("*").alias("call_date_count")) \
    .orderBy("call_date")
    #.explain()
)

In [13]:
df2.show()

+---------+---------------+
|call_date|call_date_count|
+---------+---------------+
|        1|              7|
|        2|             10|
|        3|             11|
|        4|             16|
|        5|             18|
|        6|             19|
|        7|             10|
|        8|              6|
|        9|             19|
|       10|              3|
|       11|             12|
|       12|              8|
|       13|              7|
|       14|              5|
|       15|             15|
|       16|             17|
|       17|              3|
|       18|              3|
|       19|              4|
|       20|              7|
+---------+---------------+
only showing top 20 rows


In [ ]:
df2 = (
    df \
    .withColumn("cnvs_ts",F.to_timestamp("cnvs_ts", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("mm-dd", F.date_format("cnvs_ts", "MM-dd")) \
    .groupBy("mm-dd").agg(F.count("*").alias("call_date_count")) \
    .orderBy("mm-dd") #orderBy are extremely expensive operations as they require shuffling of data across the cluster, so we should avoid them if not necessary.
    #.explain()
)

26/04/28 00:42:30 ERROR Worker: Connection to master failed! Waiting for master to reconnect...
26/04/28 00:42:30 ERROR Worker: Connection to master failed! Waiting for master to reconnect...
26/04/28 00:42:30 ERROR Worker: Connection to master failed! Waiting for master to reconnect...
26/04/28 00:42:30 ERROR Worker: Connection to master failed! Waiting for master to reconnect...
26/04/28 00:42:30 ERROR Worker: Connection to master failed! Waiting for master to reconnect...
26/04/28 00:42:30 ERROR Worker: Connection to master failed! Waiting for master to reconnect...
26/04/28 00:42:30 WARN StandaloneAppClient$ClientEndpoint: Connection to 192.168.0.101:53380 failed; waiting for master to reconnect...
26/04/28 00:42:30 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
26/04/28 00:42:30 WARN StandaloneAppClient$ClientEndpoint: Connection to 192.168.0.101:53380 failed; waiting for master to reconnect...
26/04/28 00:42:30 WARN Master: App app-2

In [15]:
df2.show(df2.count())

+-----+---------------+
|mm-dd|call_date_count|
+-----+---------------+
|01-05|              3|
|01-06|              8|
|01-07|             10|
|01-08|              2|
|01-09|             12|
|01-12|              7|
|01-13|              7|
|01-14|              5|
|01-15|              5|
|01-16|              8|
|01-19|              2|
|01-20|              7|
|01-21|             13|
|01-22|             12|
|01-23|              9|
|01-26|              3|
|01-27|              2|
|01-28|             11|
|01-29|              7|
|01-30|              6|
|02-02|              4|
|02-03|              4|
|02-04|              8|
|02-05|             10|
|02-06|             11|
|12-01|              7|
|12-02|              6|
|12-03|              7|
|12-04|              8|
|12-05|              5|
|12-08|              4|
|12-09|              7|
|12-10|              3|
|12-11|             12|
|12-12|              1|
|12-15|             10|
|12-16|              9|
|12-17|              3|
|12-18|         